In [1]:
import os # Configure which GPU

if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
    os.environ["CUDA_VISIBLE_DEVICES"] = "2"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Import Sionna
try:
    import sionna
except ImportError as e:
    # Install Sionna if package is not already installed
    import os
    os.system("pip install sionna")
    import sionna

# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

# Colab does currently not support the latest version of ipython.
# Thus, the preview does not work in Colab. However, whenever possible we
# strongly recommend to use the scene preview mode.
try: # detect if the notebook runs in Colab
    import google.colab
    no_preview = True # deactivate preview
except:
    if os.getenv("SIONNA_NO_PREVIEW"):
        no_preview = True
    else:
        no_preview = False

resolution = [480,320] # increase for higher quality of renderings

# Define magic cell command to skip a cell if needed
from IPython.core.magic import register_cell_magic
from IPython import get_ipython

@register_cell_magic
def skip_if(line, cell):
    if eval(line):
        return
    get_ipython().run_cell(cell)

# Set random seed for reproducibility
sionna.config.seed = 42

In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import time

# Import Sionna RT components
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, Camera

# For link-level simulations
from sionna.channel import cir_to_ofdm_channel, subcarrier_frequencies, OFDMChannel, ApplyOFDMChannel, CIRDataset
from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver
from sionna.utils import compute_ber, ebnodb2no, PlotBER

from nr_pusch.model import Model, CIRGenerator, PUSCHModel
from nr_pusch.configs import SISOConfig, MIMOConfig


In [3]:
siso_configuration = SISOConfig(
    batch_size=1,
    num_rx=1,
    num_rx_ant=2,
    num_tx=1,
    num_tx_ant=1,
    subcarrier_spacing=15e3,
    num_time_steps=14
)
# System parameters
subcarrier_spacing = 30e3
num_time_steps = 14 # Total number of ofdm symbols per slot

num_tx = 4 # Number of users
num_rx = 1 # Only one receiver considered
num_tx_ant = 4 # Each user has 4 antennas
num_rx_ant = 16 # The receiver is equipped with 16 antennas

# batch_size for CIR generation
batch_size_cir = 500
batch_size=10
mimo_configuration = MIMOConfig(
    num_prb=16,
    mcs_index=14,
    num_layers=1,
    mcs_table=1,
    domain="freq",
    num_tx_ant=num_tx_ant,
    num_tx=num_tx,
    subcarrier_spacing=subcarrier_spacing,
    num_rx_ant=num_rx_ant,
    num_rx=num_rx,
    num_symbols=num_time_steps,
    batch_size=batch_size,
    num_paths=75
)

In [ ]:
import os
import numpy as np
import pandas as pd

# Define the number of samples per scenario and model type
num_samples_per_scenario = 2000

# Initialize the dataset
dataset = []

# Define the scenarios and Eb/No values
scenarios = ["A", "B", "C", "D", "E"]
ebno_values = range(-3, 19, 2)
model_types = ["tdl", "cdl"]

# Loop through each model type
# for scenario in ["A", "B", "C", "D", "E"]:
#     for ebno_db in range(-3, 19, 2):
#         model_tdl = Model(
#             perfect_csi=False,
#             detector=None,
#             config=siso_configuration,
#             model_type="tdl",
#             scenario=scenario,
#         )
#         # make tdl_scenario as key
#         features[f"tdl_{scenario}_{ebno_db}"] = model.extract_features(ebno_db=ebno_db)

#         model = Model(
#             perfect_csi=False,
#             detector=None,
#             config=siso_configuration,
#             model_type="cdl",
#             scenario=scenario,
#         )
#         # make cdl_scenario as key
#         features[f"cdl_{scenario}_{ebno_db}"] = model.extract_features(ebno_db=ebno_db)



# Define the number of samples per scenario and model type
num_samples_per_scenario = 2000

# Initialize the dataset


# Define the scenarios and Eb/No values
scenarios = ["A", "B", "C", "D", "E"]
ebno_min = -3
ebno_max = 19
model_types = ["tdl", "cdl"]

# Loop through each model type
for model_type in model_types:
    # Loop through each scenario
    for scenario in scenarios:
        model = Model(
            perfect_csi=False,
            detector=None,
            config=mimo_configuration,
            model_type=model_type,
            scenario=scenario,
        )
        # Loop through the number of samples per scenario
        for sample_idx in range(num_samples_per_scenario):
            ebno_db = np.random.uniform(ebno_min, ebno_max)
            print(f"Generating sample {sample_idx + 1}/{num_samples_per_scenario} for scenario {scenario} and model type {model_type}")

            # Randomly select an Eb/No value (Monte Carlo sampling)

            # Extract features (Simulation)
            features = model.extract_features(ebno_db=ebno_db)

            # Add additional metadata
            features["scenario"] = scenario
            features["ebno_db"] = ebno_db
            features["model_type"] = model_type
            if scenario == "A" or scenario == "B" or scenario == "C":
                features["label"] = "nlos"
            else:
                features["label"] = "los"
            # Append the features to the dataset (Aggregation)
            dataset.append(features)

# Convert the dataset to a structured format (e.g., Pandas DataFrame)
df = pd.DataFrame(dataset)

# Save the dataset to a file (e.g., CSV)
df.to_csv("dataset.csv", index=False)

# Convert to HDF5
df.to_hdf("dataset.h5", key="data", mode='w')

print("Dataset generation complete. Saved to dataset.csv and dataset.h5")

Generating sample 1/2000 for scenario A and model type tdl
Generating sample 2/2000 for scenario A and model type tdl
Generating sample 3/2000 for scenario A and model type tdl
Generating sample 4/2000 for scenario A and model type tdl
Generating sample 5/2000 for scenario A and model type tdl
Generating sample 6/2000 for scenario A and model type tdl
Generating sample 7/2000 for scenario A and model type tdl
Generating sample 8/2000 for scenario A and model type tdl
Generating sample 9/2000 for scenario A and model type tdl
Generating sample 10/2000 for scenario A and model type tdl
Generating sample 11/2000 for scenario A and model type tdl
Generating sample 12/2000 for scenario A and model type tdl
Generating sample 13/2000 for scenario A and model type tdl
Generating sample 14/2000 for scenario A and model type tdl
Generating sample 15/2000 for scenario A and model type tdl
Generating sample 16/2000 for scenario A and model type tdl
Generating sample 17/2000 for scenario A and mode